###Run Shared Libraries

In [0]:
%run ../Misc/sharedlibraries

In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimcostcenter"

###Read Bronze tables

In [0]:
costcenterDf = spark.table("bronze.costcenter")

In [0]:
costcenterDf.printSchema()

###Build Dimension/Fact table

In [0]:
dimcostcenterDf = costcenterDf.filter(costcenterDf.RecordId.isNotNull()
    ).select(
        costcenterDf.CostCenterNumber,
        F.when(costcenterDf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(costcenterDf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
        F.from_utc_timestamp(costcenterDf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
        costcenterDf.Vat,
        costcenterDf.RecordId.alias("CostCenterRecordId")
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("CostCenterHashKey", F.xxhash64("CostCenterRecordId")
    )

display(dimcostcenterDf)

###Final dataframe

In [0]:
df_final = dimcostcenterDf

## Write to Silver Schema

In [0]:
saveDeltaTableToCatalog(df_final,"silver",Entity)